#  Phân Tích Đánh Giá Khách Sạn Bằng Apache Pig

### Mô tả dữ liệu

| Cột | Tên | Mô tả | Ví dụ |
|-----|-----|-------|-------|
| 1 | `id` | Mã bình luận | 1, 2, 3... |
| 2 | `review` | Nội dung bình luận | "Phòng sạch sẽ, nhân viên thân thiện" |
| 3 | `category` | Phân loại đánh giá | GENERAL, QUALITY, DESIGN&FEATURES... |
| 4 | `aspect` | Khía cạnh đánh giá | HOTEL, SERVICE, ROOMS, FOOD&DRINKS... |
| 5 | `sentiment` | Cảm xúc | positive, negative, neutral |

**Phân cách:** dấu chấm phẩy (`;`)

### Mục tiêu
1. Tiền xử lý dữ liệu văn bản tiếng Việt
2. Thống kê tần số từ, bình luận theo category và aspect
3. Xác định khía cạnh tích cực/tiêu cực nhất
4. Tìm top từ tích cực/tiêu cực theo từng phân loại
5. Tìm top từ liên quan nhất theo từng phân loại


In [12]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

pig --version 2>&1 | grep "Apache Pig version"
echo "Pig OK!"


Apache Pig version 0.17.0 (r1797386) 
Pig OK!


## Bài 1:

Thực hiện các thao tác sau đối với bộ dữ liệu:
- Đưa tất cả ký tự về chữ thường (lowercase).
- Tách các dòng bình luận thành dãy các từ (từ được tách ra từ câu theo khoảng trắng).
- Loại bỏ các stop word (dựa vào danh sách ```stopword.txt```)

**Phương pháp:**
1. `LOWER()` — chuyển toàn bộ review về chữ thường
2. `TOKENIZE()` + `FLATTEN()` — tách câu thành từng từ riêng biệt
3. `LEFT JOIN` với bảng stopwords → `FILTER` loại bỏ từ trùng khớp


In [ ]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02

rm -rf Output/BaiTap1

cat > bai1.pig <<'EOF'
raw = LOAD 'Data/hotel-review.csv' USING PigStorage(';') AS (id:int, review:chararray, category:chararray, aspect:chararray, sentiment:chararray);
stopwords = LOAD 'Data/stopwords.txt' AS (word:chararray);

tokens = FOREACH raw GENERATE id, FLATTEN(TOKENIZE(LOWER(review), ' ,.:;!?()[]{}')) AS word, category, aspect, sentiment;
tokens_clean = FILTER tokens BY TRIM(word) != '';

stops = FOREACH stopwords GENERATE LOWER(TRIM(word)) AS stopword;
joined = JOIN tokens_clean BY word LEFT OUTER, stops BY stopword;
filtered = FILTER joined BY stops::stopword IS NULL;

bai1_out = FOREACH filtered GENERATE tokens_clean::id, tokens_clean::word, tokens_clean::category, tokens_clean::aspect, tokens_clean::sentiment;
STORE bai1_out INTO 'Output/BaiTap1' USING PigStorage('\t');
EOF

In [ ]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02

pig -x local bai1.pig 2>&1 | tail -5

echo "10 DÒNG ĐẦU KẾT QUẢ BÀI 1 "
head -10 Output/BaiTap1/part-r-00000

2026-04-05 10:42:13,858 [main] WARN  org.apache.hadoop.metrics2.impl.MetricsSystemImpl - JobTracker metrics system already initialized!
2026-04-05 10:42:13,862 [main] WARN  org.apache.hadoop.metrics2.impl.MetricsSystemImpl - JobTracker metrics system already initialized!
2026-04-05 10:42:13,864 [main] WARN  org.apache.hadoop.metrics2.impl.MetricsSystemImpl - JobTracker metrics system already initialized!
2026-04-05 10:42:13,877 [main] INFO  org.apache.pig.backend.hadoop.executionengine.mapReduceLayer.MapReduceLauncher - Success!
2026-04-05 10:42:13,915 [main] INFO  org.apache.pig.Main - Pig script completed in 11 seconds and 861 milliseconds (11861 ms)
10 DÒNG ĐẦU KẾT QUẢ BÀI 1 
6752	"	QUALITY	FOOD&DRINKS	negative
4006	"	DESIGN&FEATURES	HOTEL	positive
6752	"	STYLE&OPTIONS	FOOD&DRINKS	negative
4152	"	MISCELLANEOUS	ROOM_AMENITIES	negative
4894	"	GENERAL	ROOM_AMENITIES	positive
5873	"	COMFORT	ROOMS	positive
5873	"	GENERAL	ROOMS	positive
4894	"	GENERAL	LOCATION	positive
1819	"	GENERAL	LOCA

---
##  Bài 2: Thống kê

**Yêu cầu:**
- Thống kê tần số xuất hiện của các từ → Chỉ ra các từ xuất hiện trên 500 lần
- Thống kê số bình luận theo từng phân loại (category)
- Thống kê số bình luận theo từng khía cạnh đánh giá (aspect)

**Phương pháp:**
- `GROUP BY` + `COUNT()` — nhóm và đếm số lượng
- `ORDER BY ... DESC` — sắp xếp giảm dần
- `FILTER ... BY freq > 500` — lọc từ xuất hiện trên 500 lần


In [ ]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02
rm -rf Output/BaiTap2_*

cat > bai2.pig <<'EOF'
raw = LOAD 'Data/hotel-review.csv' USING PigStorage(';') AS (
    id:int, 
    review:chararray, 
    category:chararray, 
    aspect:chararray, 
    sentiment:chararray);
bai1 = LOAD 'Output/BaiTap1' USING PigStorage('\t') AS (
    id:int, 
    word:chararray, 
    category:chararray, 
    aspect:chararray, 
    sentiment:chararray);

-- 2a. Thống kê từ xuất hiện > 500 lần
grp_word = GROUP bai1 BY word;
word_freq = FOREACH grp_word GENERATE group AS word, COUNT(bai1) AS freq;
STORE (ORDER (FILTER word_freq BY freq > 500) BY freq DESC) INTO 'Output/BaiTap2_WordFreq' USING PigStorage('\t');

-- 2b. Thống kê theo Category
grp_cat = GROUP raw BY category;
cat_count = FOREACH grp_cat GENERATE group AS category, COUNT(raw) AS freq;
STORE (ORDER cat_count BY freq DESC) INTO 'Output/BaiTap2_Category' USING PigStorage('\t');

-- 2c. Thống kê theo Aspect
grp_asp = GROUP raw BY aspect;
asp_count = FOREACH grp_asp GENERATE group AS aspect, COUNT(raw) AS freq;
STORE (ORDER asp_count BY freq DESC) INTO 'Output/BaiTap2_Aspect' USING PigStorage('\t');
EOF

In [ ]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02

pig -x local bai2.pig 2>&1 | tail -5

echo "  2a. CÁC TỪ XUẤT HIỆN TRÊN 500 LẦN"
echo ""
printf "%-30s %s\n" "TỪ" "TẦN SỐ"
echo "----------------------------------------------"

cat Output/BaiTap2_WordFreq/part-r-00000 | while IFS=$'\t' read word freq; do
    printf "%-30s %s\n" "$word" "$freq"
done

echo ""
echo "  2b. SỐ BÌNH LUẬN THEO CATEGORY"
echo ""
printf "%-25s %s\n" "CATEGORY" "SỐ BÌNH LUẬN"
echo "----------------------------------------------"

cat Output/BaiTap2_Category/part-r-00000 | while IFS=$'\t' read cat total; do
    printf "%-25s %s\n" "$cat" "$total"
done

echo ""
echo "  2c. SỐ BÌNH LUẬN THEO ASPECT"
echo ""
printf "%-25s %s\n" "ASPECT" "SỐ BÌNH LUẬN"
echo "----------------------------------------------"

cat Output/BaiTap2_Aspect/part-r-00000 | while IFS=$'\t' read asp total; do
    printf "%-25s %s\n" "$asp" "$total"
done

2026-04-05 10:42:45,398 [main] WARN  org.apache.hadoop.metrics2.impl.MetricsSystemImpl - JobTracker metrics system already initialized!
2026-04-05 10:42:45,402 [main] WARN  org.apache.hadoop.metrics2.impl.MetricsSystemImpl - JobTracker metrics system already initialized!
2026-04-05 10:42:45,404 [main] WARN  org.apache.hadoop.metrics2.impl.MetricsSystemImpl - JobTracker metrics system already initialized!
2026-04-05 10:42:45,407 [main] INFO  org.apache.pig.backend.hadoop.executionengine.mapReduceLayer.MapReduceLauncher - Success!
2026-04-05 10:42:45,451 [main] INFO  org.apache.pig.Main - Pig script completed in 14 seconds and 828 milliseconds (14828 ms)
  2a. CÁC TỪ XUẤT HIỆN TRÊN 500 LẦN

TỪ                           TẦN SỐ
----------------------------------------------
phòng                         6773
khách                         5201
sạn                          4061
vụ                           3816
tiện                         3284
không                         3268
nhân        

---
##  Bài 3: Khía cạnh tích cực nhất & tiêu cực nhất

**Yêu cầu:**  
Xác định khía cạnh (aspect) nào nhận nhiều đánh giá tiêu cực (negative) nhất, 
và khía cạnh nào nhận nhiều đánh giá tích cực (positive) nhất.

**Phương pháp:**
1. `FILTER` lọc theo sentiment (positive / negative)
2. `GROUP BY aspect` + `COUNT()` — đếm số bình luận mỗi aspect
3. `ORDER BY ... DESC` + `LIMIT 1` — lấy aspect có số lượng cao nhất


In [ ]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02
rm -rf Output/BaiTap3_*

cat > bai3.pig <<'EOF'

raw = LOAD 'Data/hotel-review.csv' USING PigStorage(';') 
      AS (id:int, review:chararray, category:chararray, aspect:chararray, sentiment:chararray);

-- 3a. SẮP XẾP SỐ ĐÁNH GIÁ NEGATIVE THEO ASPECT
negative_reviews = FILTER raw BY sentiment == 'negative';
neg_grouped = GROUP negative_reviews BY aspect;
neg_count = FOREACH neg_grouped GENERATE group AS aspect, COUNT(negative_reviews) AS neg_total;
neg_sorted = ORDER neg_count BY neg_total DESC;

-- Chỉ cần lưu danh sách tổng đã sắp xếp
STORE neg_sorted INTO 'Output/BaiTap3_NegativeAll' USING PigStorage('\t');

-- 3b. SẮP XẾP SỐ ĐÁNH GIÁ POSITIVE THEO ASPECT
positive_reviews = FILTER raw BY sentiment == 'positive';
pos_grouped = GROUP positive_reviews BY aspect;
pos_count = FOREACH pos_grouped GENERATE group AS aspect, COUNT(positive_reviews) AS pos_total;
pos_sorted = ORDER pos_count BY pos_total DESC;

STORE pos_sorted INTO 'Output/BaiTap3_PositiveAll' USING PigStorage('\t');
EOF

In [ ]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))


cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02

pig -x local bai3.pig 2>&1 | tail -5

echo "   ASPECT CÓ NHIỀU ĐÁNH GIÁ TIÊU CỰC NHẤT"

head -n 1 Output/BaiTap3_NegativeAll/part-r-00000 | while IFS=$'\t' read asp total; do
    printf "%s — %s bình luận negative\n" "$asp" "$total"
done

echo ""
echo "Chi tiết số lượng negative theo từng aspect:"
echo ""
printf "%-25s %s\n" "ASPECT" "SỐ NEGATIVE"
echo "----------------------------------------------"
cat Output/BaiTap3_NegativeAll/part-r-00000 | while IFS=$'\t' read asp total; do
    printf "%-25s %s\n" "$asp" "$total"
done

echo ""
echo "   ASPECT CÓ NHIỀU ĐÁNH GIÁ TÍCH CỰC NHẤT"
head -n 1 Output/BaiTap3_PositiveAll/part-r-00000 | while IFS=$'\t' read asp total; do
    printf "%s — %s bình luận positive\n" "$asp" "$total"
done

echo ""
echo "Chi tiết số lượng positive theo từng aspect:"
echo ""
printf "%-25s %s\n" "ASPECT" "SỐ POSITIVE"
echo "----------------------------------------------"
cat Output/BaiTap3_PositiveAll/part-r-00000 | while IFS=$'\t' read asp total; do
    printf "%-25s %s\n" "$asp" "$total"
done

2026-04-05 10:43:09,028 [main] WARN  org.apache.hadoop.metrics2.impl.MetricsSystemImpl - JobTracker metrics system already initialized!
2026-04-05 10:43:09,033 [main] WARN  org.apache.hadoop.metrics2.impl.MetricsSystemImpl - JobTracker metrics system already initialized!
2026-04-05 10:43:09,035 [main] WARN  org.apache.hadoop.metrics2.impl.MetricsSystemImpl - JobTracker metrics system already initialized!
2026-04-05 10:43:09,043 [main] INFO  org.apache.pig.backend.hadoop.executionengine.mapReduceLayer.MapReduceLauncher - Success!
2026-04-05 10:43:09,089 [main] INFO  org.apache.pig.Main - Pig script completed in 9 seconds and 524 milliseconds (9524 ms)
   ASPECT CÓ NHIỀU ĐÁNH GIÁ TIÊU CỰC NHẤT

ROOMS — 515 bình luận negative

Chi tiết số lượng negative theo từng aspect:

ASPECT                    SỐ NEGATIVE
----------------------------------------------
ROOMS                     515
ROOM_AMENITIES            403
HOTEL                     341
FOOD&DRINKS               277
SERVICE        

---
### Bài 4: Top 5 từ tích cực / tiêu cực theo từng category

**Yêu cầu:**
- Dựa vào từng phân loại bình luận, xác định **5 từ** mang ý nghĩa tích cực nhất
- Dựa vào từng phân loại bình luận, xác định **5 từ** mang ý nghĩa tiêu cực nhất

**Phương pháp:**
- Lọc theo sentiment → Nhóm theo `(category, word)` → Đếm tần số
- Trong mỗi category, sắp xếp giảm dần và lấy top 5 bằng **Nested Block** (`ORDER` + `LIMIT` bên trong `FOREACH`)

**Ý tưởng:** Từ nào xuất hiện nhiều nhất trong bình luận positive/negative 
của 1 category → từ đó mang tính tích cực/tiêu cực nhất cho category đó.


In [24]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02

cat > Source_Pig/bai4.pig <<'EOF'

-- BÀI 4: TOP 5 TỪ TÍCH CỰC / TIÊU CỰC THEO CATEGORY

%declare INPUT_PATH 'Output/BaiTap1'
%declare POS_OUTPUT 'Output/BaiTap4_top5_positive'
%declare NEG_OUTPUT 'Output/BaiTap4_top5_negative'

data = LOAD '$INPUT_PATH' USING PigStorage('\t') 
       AS (id:int, word:chararray, category:chararray, aspect:chararray, sentiment:chararray);

-- 4A. POSITIVE
pos_words  = FILTER data BY sentiment == 'positive';
pos_counts = FOREACH (GROUP pos_words BY (category, word)) 
             GENERATE FLATTEN(group) AS (category, word), COUNT(pos_words) AS freq;
pos_top5   = FOREACH (GROUP pos_counts BY category) {
    sorted = ORDER pos_counts BY freq DESC;
    top5   = LIMIT sorted 5;
    GENERATE FLATTEN(top5);
};
STORE pos_top5 INTO '$POS_OUTPUT' USING PigStorage('\t');

-- 4B. NEGATIVE
neg_words  = FILTER data BY sentiment == 'negative';
neg_counts = FOREACH (GROUP neg_words BY (category, word)) 
             GENERATE FLATTEN(group) AS (category, word), COUNT(neg_words) AS freq;
neg_top5   = FOREACH (GROUP neg_counts BY category) {
    sorted = ORDER neg_counts BY freq DESC;
    top5   = LIMIT sorted 5;
    GENERATE FLATTEN(top5);
};
STORE neg_top5 INTO '$NEG_OUTPUT' USING PigStorage('\t');

EOF

In [ ]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02
rm -rf Output/BaiTap4_top5_positive
rm -rf Output/BaiTap4_top5_negative

pig -x local Source_Pig/bai4.pig 2>&1 | tail -5

echo ""
echo "TOP 5 TỪ TÍCH CỰC THEO CATEGORY"
printf "%-25s %-20s %s\n" "CATEGORY" "TỪ" "TẦN SỐ"
echo "----------------------------------------------------"
cat Output/BaiTap4_top5_positive/part-r-00000 | while IFS=$'\t' read cat word freq; do
    printf "%-25s %-20s %s\n" "$cat" "$word" "$freq"
done

echo ""
echo "TOP 5 TỪ TIÊU CỰC THEO CATEGORY"
printf "%-25s %-20s %s\n" "CATEGORY" "TỪ" "TẦN SỐ"
echo "----------------------------------------------------"
cat Output/BaiTap4_top5_negative/part-r-00000 | while IFS=$'\t' read cat word freq; do
    printf "%-25s %-20s %s\n" "$cat" "$word" "$freq"
done

2026-04-05 10:51:56,524 [main] WARN  org.apache.hadoop.metrics2.impl.MetricsSystemImpl - JobTracker metrics system already initialized!
2026-04-05 10:51:56,527 [main] WARN  org.apache.hadoop.metrics2.impl.MetricsSystemImpl - JobTracker metrics system already initialized!
2026-04-05 10:51:56,529 [main] WARN  org.apache.hadoop.metrics2.impl.MetricsSystemImpl - JobTracker metrics system already initialized!
2026-04-05 10:51:56,534 [main] INFO  org.apache.pig.backend.hadoop.executionengine.mapReduceLayer.MapReduceLauncher - Success!
2026-04-05 10:51:56,568 [main] INFO  org.apache.pig.Main - Pig script completed in 12 seconds and 330 milliseconds (12330 ms)

=== TOP 5 TỪ TÍCH CỰC THEO CATEGORY ===
CATEGORY                  TỪ                 TẦN SỐ
----------------------------------------------------
PRICES                    giá                 602
PRICES                    cả                 412
PRICES                    hợp                320
PRICES                    vụ                 

## Bài 5: Top 5 từ liên quan nhất theo từng category
Dựa vào từng phân loại bình luận, xác định **5 từ** liên quan nhất 
(từ xuất hiện nhiều nhất trong category đó, không phân biệt sentiment).


In [1]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02

cat > Source_Pig/bai5.pig <<'EOF'

-- BÀI 5: TOP 5 TỪ LIÊN QUAN NHẤT THEO CATEGORY

bai1 = LOAD 'Output/BaiTap1' USING PigStorage('\t') 
       AS (id:int, word:chararray, category:chararray, aspect:chararray, sentiment:chararray);

aw_g   = GROUP bai1 BY (category, word);
aw_c   = FOREACH aw_g GENERATE FLATTEN(group) AS (category, word), COUNT(bai1) AS freq;
aw_cat = GROUP aw_c BY category;
aw_top5 = FOREACH aw_cat {
    s = ORDER aw_c BY freq DESC;
    t = LIMIT s 5;
    GENERATE FLATTEN(t);
};

STORE aw_top5 INTO 'Output/BaiTap5_Top5_Related' USING PigStorage('\t');

EOF

In [2]:
%%bash
export PIG_HOME=/home/nbl2005/pig-0.17.0
export PATH=$PATH:$PIG_HOME/bin
export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which java))))

cd /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02
rm -rf Output/BaiTap5_Top5_Related

pig -x local Source_Pig/bai5.pig 2>&1 | tail -5

echo ""
echo "TOP 5 TỪ LIÊN QUAN NHẤT THEO CATEGORY"
printf "%-25s %-20s %s\n" "CATEGORY" "TỪ" "TẦN SỐ"
echo "----------------------------------------------------"
cat Output/BaiTap5_Top5_Related/part-r-00000 | while IFS=$'\t' read cat word freq; do
    printf "%-25s %-20s %s\n" "$cat" "$word" "$freq"
done

2026-04-06 02:52:14,024 [main] WARN  org.apache.hadoop.metrics2.impl.MetricsSystemImpl - JobTracker metrics system already initialized!
2026-04-06 02:52:14,030 [main] WARN  org.apache.hadoop.metrics2.impl.MetricsSystemImpl - JobTracker metrics system already initialized!
2026-04-06 02:52:14,032 [main] WARN  org.apache.hadoop.metrics2.impl.MetricsSystemImpl - JobTracker metrics system already initialized!
2026-04-06 02:52:14,038 [main] INFO  org.apache.pig.backend.hadoop.executionengine.mapReduceLayer.MapReduceLauncher - Success!
2026-04-06 02:52:14,069 [main] INFO  org.apache.pig.Main - Pig script completed in 10 seconds and 344 milliseconds (10344 ms)

TOP 5 TỪ LIÊN QUAN NHẤT THEO CATEGORY
CATEGORY                  TỪ                 TẦN SỐ
----------------------------------------------------
PRICES                    giá                 697
PRICES                    cả                 452
PRICES                    vụ                 339
PRICES                    hợp                33

In [1]:
%%bash

echo " THỰC HÀNH PIG - PHÂN TÍCH DỮ LIỆU HOTEL REVIEW"
echo " Sinh viên: Nguyễn Bá Long - MSSV: 23520880"
echo " User hệ thống: $USER@$(hostname)"
echo " Đường dẫn hiện tại: $(pwd)"

echo " ĐÃ LƯU THÀNH CÔNG TẤT CẢ KẾT QUẢ VÀO FOLDER RESULT!"
echo -e "\n Danh sách các file kết quả:"

# Chỉ liệt kê tên file và dung lượng, KHÔNG in nội dung bên trong
ls -lh /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02/Result/

 THỰC HÀNH PIG - PHÂN TÍCH DỮ LIỆU HOTEL REVIEW
 Sinh viên: Nguyễn Bá Long - MSSV: 23520880
 User hệ thống: nbl2005@AiCiCi
 Đường dẫn hiện tại: /home/nbl2005/DS200.Q21.1.LAB/DS200.Q21.1.Lab02/Notebook
 ĐÃ LƯU THÀNH CÔNG TẤT CẢ KẾT QUẢ VÀO FOLDER RESULT!

 Danh sách các file kết quả:
total 6.5M
-rw-r--r-- 1 nbl2005 nbl2005 6.4M Apr  6 03:11 Bai1_TienXuLy.txt
-rw-r--r-- 1 nbl2005 nbl2005   99 Apr  6 03:11 Bai2a_Aspect.txt
-rw-r--r-- 1 nbl2005 nbl2005  123 Apr  6 03:11 Bai2b_Category.txt
-rw-r--r-- 1 nbl2005 nbl2005  839 Apr  6 03:11 Bai2c_WordFreq.txt
-rw-r--r-- 1 nbl2005 nbl2005   98 Apr  6 03:11 Bai3a_PositiveAll.txt
-rw-r--r-- 1 nbl2005 nbl2005   94 Apr  6 03:11 Bai3b_NegativeAll.txt
-rw-r--r-- 1 nbl2005 nbl2005  840 Apr  6 03:11 Bai4a_Top5_Pos.txt
-rw-r--r-- 1 nbl2005 nbl2005  834 Apr  6 03:11 Bai4b_Top5_Neg.txt
-rw-r--r-- 1 nbl2005 nbl2005  857 Apr  6 03:11 Bai5_Top5_Related.txt
